# SIH26083: Biometeorological Thermal Stress Demonstration
## Moving From *"What the Weather Is"* to *"What the Weather Does to Humans"*

**Problem Statement:** SIH26083 - Extreme Heatwave Early Warning and Human Thermal Stress Index  
**Organization:** Ministry of Earth Sciences (MoES) / NCMRWF  

This notebook demonstrates why **dry-bulb air temperature ($T_a$) alone fails as a public-health warning indicator**, and validates our physiological **Universal Thermal Climate Index (UTCI)** and **Wet Bulb Globe Temperature (WBGT)** models.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from backend.app.thermal.utci import calculate_utci, classify_utci
from backend.app.thermal.wbgt import calculate_wbgt, classify_wbgt
from backend.app.thermal.heat_index import calculate_heat_index
from backend.app.thermal.hazard import calculate_thermal_hazard
from backend.app.risk.engine import HeatRiskEngine

### 1. The Core Demonstration: Identical 40°C Air Temperature with Contrasting Humidity & Wind

In [ ]:
# Scenario A: Hot, Dry & Windy (15% RH, 5.0 m/s wind, 150 W/m² solar)
scen_a = calculate_thermal_hazard(temp_c=40.0, relative_humidity_pct=15.0, wind_speed_10m_m_s=5.0, solar_radiation_w_m2=150.0)

# Scenario B: Hot, Humid & Stagnant (70% RH, 0.8 m/s wind, 800 W/m² solar)
scen_b = calculate_thermal_hazard(temp_c=40.0, relative_humidity_pct=70.0, wind_speed_10m_m_s=0.8, solar_radiation_w_m2=800.0)

df_comparison = pd.DataFrame([
    {
        'Scenario': 'Scenario A (Dry & Windy)',
        'Air Temp (°C)': 40.0,
        'Relative Humidity (%)': 15.0,
        'Wind Speed (m/s)': 5.0,
        'UTCI (°C)': scen_a['metrics']['utci']['value_c'],
        'UTCI Stress': scen_a['metrics']['utci']['category'],
        'WBGT (°C)': scen_a['metrics']['wbgt']['value_c'],
        'WBGT Risk': scen_a['metrics']['wbgt']['risk_level'],
        'Hazard Score': scen_a['composite_hazard_score']
    },
    {
        'Scenario': 'Scenario B (Humid & Stagnant)',
        'Air Temp (°C)': 40.0,
        'Relative Humidity (%)': 70.0,
        'Wind Speed (m/s)': 0.8,
        'UTCI (°C)': scen_b['metrics']['utci']['value_c'],
        'UTCI Stress': scen_b['metrics']['utci']['category'],
        'WBGT (°C)': scen_b['metrics']['wbgt']['value_c'],
        'WBGT Risk': scen_b['metrics']['wbgt']['risk_level'],
        'Hazard Score': scen_b['composite_hazard_score']
    }
])

df_comparison

### 2. Parametric Sensitivity: How Humidity and Solar Radiation Escalate UTCI at Fixed 40°C

In [ ]:
rh_range = np.linspace(10, 80, 50)
utci_shade = [calculate_utci(40.0, rh, wind_speed_10m_m_s=2.0, solar_radiation_w_m2=0.0) for rh in rh_range]
utci_sun = [calculate_utci(40.0, rh, wind_speed_10m_m_s=2.0, solar_radiation_w_m2=750.0) for rh in rh_range]
wbgt_vals = [calculate_wbgt(40.0, rh, wind_speed_10m_m_s=2.0, solar_radiation_w_m2=750.0) for rh in rh_range]

plt.figure(figsize=(10, 6))
plt.plot(rh_range, utci_sun, label='UTCI (Direct Sun 750 W/m²)', color='#ef4444', linewidth=2.5)
plt.plot(rh_range, utci_shade, label='UTCI (Shade 0 W/m²)', color='#f97316', linewidth=2, linestyle='--')
plt.plot(rh_range, wbgt_vals, label='Occupational WBGT (Sun)', color='#38bdf8', linewidth=2)
plt.axhline(46.0, color='#7f0000', linestyle=':', label='UTCI Extreme Heat Stress Threshold (46°C)')
plt.axhline(32.0, color='#dc2626', linestyle=':', label='NIOSH WBGT Extreme Danger Threshold (32°C)')

plt.title('Physiological Strain Escalation at Fixed Air Temp Ta = 40°C', fontsize=14, fontweight='bold')
plt.xlabel('Relative Humidity (%)', fontsize=12)
plt.ylabel('Thermal Index (°C)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left')
plt.show()